# **🐶 Multi-class Dog Breed Classification (Using TensorFlow 2.21.0 and Transfer Learning)**

This notebook builds and end-to-end multi-class image classifier using `TensorFlow 2.21.0` and `TensorFlow Hub`.

---

## **Some things to know**

### **1. Deep Learning**
Machine learning where the model **learns its own features** through many stacked layers (a neural network), instead of you hand-picking features manually.

"Deep" = many layers.

**Types of problem we have:**
1. Classification
2. Sequence to Sequece (seq2seq) (like audio)
3. Object detection, etc..

---

### **2. Neural Network (NN)**
A stack of layers, each made of neurons.

**Three layer types:**
- **Input layer** — passes your data in. 5 features = 5 neurons. No math here.
- **Hidden layer(s)** — where learning actually happens. Can have 1 or many.
- **Output layer** — the prediction. 1 neuron for regression, N neurons for N classes.

**What one neuron does:**
```
output = (x1 × w1) + (x2 × w2) + (x3 × w3) + ...
```
Each input is **multiplied** by a weight, then summed up.

**Weight** = a number that controls how much influence one input has.
- `w = 0.0` → ignore this input
- `w = 2.0` → this input matters a lot
- `w = -0.5` → this input pushes the decision the other way

**Learning = slowly adjusting all weights until predictions get accurate.**

![Neural Network](./assets/images/diagram2_nn_anatomy.png)

---

### **3. TensorFlow**
Google's framework for building and training neural networks.

It handles for you: matrix math, gradients, GPU acceleration, saving models.

You write the architecture. TensorFlow does the heavy lifting.

---

### **4. TensorFlow Hub**
A library of **pre-trained models** hosted by Google.

Instead of training from scratch (days + massive GPU), you grab a model that's already trained on millions of images and reuse it.

---

## **5. Transfer Learning**
Using a pre-trained model as a starting point for your own task.

```
TensorHub model (already trained)
        ↓
Freeze the base layers (keep what it learned)
        ↓
Add your own output layer on top
        ↓
Train only your new layers on your data
        ↓
Done — in minutes, not days
```

**Analogy:** A doctor who studied medicine for 10 years does a 6-month cardiology specialization. They don't relearn biology — they transfer existing knowledge.

![Analogy DL and tensorflow overview](./assets/images/diagram1_overview.png)

---

### **The Big Picture**

```
Your data
   → TensorFlow (the tool)
      → Neural Network (the model)
         → Pre-trained weights from TensorHub
            → Fine-tuned with Transfer Learning
               → Your prediction
```

---

> **Rule of thumb:** In real projects, you almost never train from scratch.
> Grab a model from TensorHub → freeze it → add your layer → train. That's the job.

---

**Back to our mental model:**

![ml-project-overview](./assets/images/matplotlib-6-step-ml-framework-tools-matplotlib-highlight.png)

Also we have **Google collab** in area space of jupyter notebook.

---

<div style="background-color: white; display: inline-block; padding: 8px;">
  <img src="./assets/images/unstructured-data-tesnorflow-workflow.png" alt="tensor-workflow" />
</div>

## **1. Problem defination**

Identifying the breed of the dog given the image of the dog.

When I'm sitting at the cafe and I take photo of a dog, I want to know that what breed of the dog it is.

---

## **2. Data**

The data we're using is from [Kaggle's Dog Breed Identification Competition](https://www.kaggle.com/competitions/dog-breed-identification/data).

---

## **3. Evaluation**

The evaluation is a file with prediction probabilities for each dog breed of each test image.

[Here](www.kaggle.com/competitions/dog-breed-identification/overview/evaluation)

---

## **4. Features**

**Some information about the data:**
- We're are dealing with images (unstructred data) so it's probably we use deep learning/transfer learning.
- There are `120` breeds of dogs (this means there are 120 different classes).
- There are around `10,000+` images in the training set (these images have labels).
- There are around `10,000+` images in the test set (these images have no labels, because we'll want to predict them).


### **Get our workspace ready**

- Import TensorFlow 2.x ✅
- Import TensorFlow Hub ✅
- Make sure we're using a GPU ✅

In [1]:
# Import necesaary Tools from colab
import tensorflow as tf
import tensorflow_hub as hub

print("TF version:", tf.__version__)
print("TF Hub Version: ", hub.__version__)

# Check for GPU availability
print("GPU", "available (YES!!!!)" if tf.config.list_physical_devices("GPU") else "not available :(")

TF version: 2.19.0
TF Hub Version:  0.16.1
GPU available (YES!!!!)


**To select GPU:**
- go to `Kernal`
  - `Select another kernal`
    - `Collab`
      - `New Collab server`
        - `GPU`
          - `T4`  --> Gives you the GPU access default is CPU

In [2]:
tf.config.list_physical_devices("GPU")

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

## **Getting our data ready (turning into Tensors)**

With all machine learning models, our data has to be in numerical format. So that's what we'll be doing first.
Turning our **images** into **Tensors** (numerical representations).

Let's start by accessing our data and checking out the labels.

In [4]:
from google.colab import drive # type: ignore
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [3]:
!unzip -q "/content/drive/MyDrive/dog-breed-identification.zip" -d "/content/drive/MyDrive/dog-vision-project-3"

In [ ]:
import glob  # Provides filename pattern matching (wildcards)

base = "/content/drive/MyDrive/dog-vision-project-3"  # Root folder where dataset was extracted
print("Top folders:", glob.glob(base + "/*")[:20])  # Show first 20 items directly inside base
print("labels.csv found at:", glob.glob(base + "/**/labels.csv", recursive=True)[:10])  # Search recursively and show up to 10 labels.csv matches

Top folders: ['/content/drive/MyDrive/dog-vision-project-3/labels.csv', '/content/drive/MyDrive/dog-vision-project-3/sample_submission.csv', '/content/drive/MyDrive/dog-vision-project-3/test', '/content/drive/MyDrive/dog-vision-project-3/train']
labels.csv found at: ['/content/drive/MyDrive/dog-vision-project-3/labels.csv']


In [9]:
# Checkout the labels of our data
import pandas as pd
label_csv = pd.read_csv("/content/drive/MyDrive/dog-vision-project-3/labels.csv")
print(label_csv.describe())
print()
print(label_csv.head())

                                      id               breed
count                              10222               10222
unique                             10222                 120
top     fff43b07992508bc822f33d8ffd902ae  scottish_deerhound
freq                                   1                 126

                                 id             breed
0  000bec180eb18c7604dcecc8fe0dba07       boston_bull
1  001513dfcb2ffafc82cccf4d8bbaba97             dingo
2  001cdf01b096e06d78e9e5112d419397          pekinese
3  00214f311d5d2247d5dfe4fe24b2303d          bluetick
4  0021f9ceb3235effd7fcde7f7538ed62  golden_retriever


<details>
  <summary><b>Click to view setup summary</b></summary>

### 1. Mount Google Drive
- What we did: Connected the Colab runtime to Google Drive.
- Why: Colab GPU runs on a remote machine, so it needs Drive access to read your files.

### 2. Unzip the dataset
- What we did: Extracted dog-breed-identification.zip into your project folder on Drive.
- Why: Uploading one zip is easier, and training needs normal extracted folders and files.

### 3. Load labels.csv
- What we did: Read labels.csv using pandas from the extracted dataset path.
- Why: labels.csv maps each image to its breed, which is required for supervised training.

</details>